In [4]:
import pandas as pd
import numpy as np

In [27]:
df_test = pd.read_csv('data/test.csv')

In [28]:
df_test

,id,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,windspeed
0,2190,1,1019.5,17.5,15.8,12.7,14.9,96.0,99.0,0.0,50.0,24.3
1,2191,2,1016.5,17.5,16.5,15.8,15.1,97.0,99.0,0.0,50.0,35.3
2,2192,3,1023.9,11.2,10.4,9.4,8.9,86.0,96.0,0.0,40.0,16.9
3,2193,4,1022.9,20.6,17.3,15.2,9.5,75.0,45.0,7.1,20.0,50.6
4,2194,5,1022.2,16.1,13.8,6.4,4.3,68.0,49.0,9.2,20.0,19.4
...,...,...,...,...,...,...,...,...,...,...,...,...
725,2915,361,1020.8,18.2,17.6,16.1,13.7,96.0,95.0,0.0,20.0,34.3
726,2916,362,1011.7,23.2,18.1,16.0,16.0,78.0,80.0,1.6,40.0,25.2
727,2917,363,1022.7,21.0,18.5,17.0,15.5,92.0,96.0,0.0,50.0,21.9
728,2918,364,1014.4,21.0,20.0,19.7,19.8,94.0,93.0,0.0,50.0,39.5


In [29]:
# change day to day_repaired

df_test['day_repaired'] = df_test['day']
df_test.drop(['day'], axis=1, inplace=True)

In [54]:
def transform_circular(data, column):
    new_data = data.copy()
    new_data[column + '_sin'] = np.sin(2 * np.pi * new_data[column] / max(new_data[column]))
    new_data[column + '_cos'] = np.cos(2 * np.pi * new_data[column] / max(new_data[column]))
    return new_data

test = transform_circular(df_test, 'day_repaired')
test = transform_circular(test, 'winddirection')

test.drop(['id', 'day_repaired', 'winddirection'], axis=1, inplace=True)

In [55]:
len(test.index)

730

In [56]:
import itertools
def generate_combinations(df, exclude_columns=[]):
    selected_columns = [col for col in df.columns if col not in exclude_columns]

    new_columns = {}

    for n in range(2, 3):
        for cols in itertools.combinations(selected_columns, n):
            col_combination = '_'.join(cols)

            # Addition
            new_columns[f'{col_combination}_plus'] = df[list(cols)].sum(axis=1)

            # Subtraction
            if n == 2:
                new_columns[f'{cols[0]}_minus_{cols[1]}'] = df[cols[0]] - df[cols[1]]
                new_columns[f'{cols[1]}_minus_{cols[0]}'] = df[cols[1]] - df[cols[0]]
            else:
                for pair in itertools.combinations(cols, 2):
                    new_columns[f'{pair[0]}_minus_{pair[1]}'] = df[pair[0]] - df[pair[1]]
                    new_columns[f'{pair[1]}_minus_{pair[0]}'] = df[pair[1]] - df[pair[0]]

            # Multiplication
            new_columns[f'{col_combination}_times'] = df[list(cols)].prod(axis=1)

            # Division
            if n == 2:
                new_columns[f'{cols[0]}_div_{cols[1]}'] = df[cols[0]] / df[cols[1]]
                new_columns[f'{cols[1]}_div_{cols[0]}'] = df[cols[1]] / df[cols[0]]
            else:
                for pair in itertools.combinations(cols, 2):
                    new_columns[f'{pair[0]}_div_{pair[1]}'] = df[pair[0]] / df[pair[1]]
                    new_columns[f'{pair[1]}_div_{pair[0]}'] = df[pair[1]] / df[pair[0]]

    df_new = pd.DataFrame(new_columns, index=df.index)

    df_comb = pd.concat([df, df_new], axis=1)
    df_comb.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_comb.fillna(0, inplace=True)
    return df_comb

test = generate_combinations(test, exclude_columns=['rainfall'])

In [57]:
len(test.index)

730

In [58]:
def add_prev_days(df):

    df_prev = df.copy()

    columns_to_drop = [col for col in df.columns if 'day' in col]

    df_shifted1 = df_prev.shift(1)
    df_shifted1.drop(columns_to_drop, axis=1, inplace=True)

    df_prev = pd.concat([df_prev, df_shifted1.add_prefix("prev1_")], axis=1)

    columns_to_drop = [col for col in df_shifted1.columns if any(keyword in col for keyword in ['winddirection', 'rainfall'])]

    df_shifted1.drop(columns_to_drop, axis=1, inplace=True)

    columns_to_drop = [col for col in df.columns if any(keyword in col for keyword in ['day', 'winddirection', 'rainfall'])]

    df_shifted2 = df.shift(2)
    df_shifted2.drop(columns_to_drop, axis=1, inplace=True)

    df_shifted3 = df.shift(3)
    df_shifted3.drop(columns_to_drop, axis=1, inplace=True)

    df_avg2 = pd.concat([df_shifted1, df_shifted2]).groupby(level=0).mean()
    df_avg2[(df_shifted1.isna()) | (df_shifted2.isna())] = np.nan
    df_avg2 = df_avg2.add_prefix("avg2_")
    df_avg2 = df_avg2.round(1)

    df_avg3 = pd.concat([df_shifted1, df_shifted2, df_shifted3]).groupby(level=0).mean()
    df_avg3[(df_shifted1.isna()) | (df_shifted2.isna()) | (df_shifted3.isna())] = np.nan
    df_avg3 = df_avg3.add_prefix("avg3_")
    df_avg3 = df_avg3.round(1)

    df_prev = pd.concat([df_prev, df_avg2, df_avg3], axis=1)
    # df_prev = df_prev.dropna().reset_index(drop=False)

    return df_prev

test = add_prev_days(test)

In [59]:
len(test.index)

730

In [60]:
test

,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,windspeed,day_repaired_sin,...,avg3_windspeed_minus_cloud,avg3_cloud_windspeed_times,avg3_cloud_div_windspeed,avg3_windspeed_div_cloud,avg3_sunshine_windspeed_plus,avg3_sunshine_minus_windspeed,avg3_windspeed_minus_sunshine,avg3_sunshine_windspeed_times,avg3_sunshine_div_windspeed,avg3_windspeed_div_sunshine
0,1019.5,17.5,15.8,12.7,14.9,96.0,99.0,0.0,24.3,1.721336e-02,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1016.5,17.5,16.5,15.8,15.1,97.0,99.0,0.0,35.3,3.442161e-02,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1023.9,11.2,10.4,9.4,8.9,86.0,96.0,0.0,16.9,5.161967e-02,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1022.9,20.6,17.3,15.2,9.5,75.0,45.0,7.1,50.6,6.880243e-02,...,-72.5,2507.6,4.2,0.3,25.5,-25.5,25.5,0.0,0.0,0.0
4,1022.2,16.1,13.8,6.4,4.3,68.0,49.0,9.2,19.4,8.596480e-02,...,-45.7,2464.7,3.1,0.6,36.6,-31.9,31.9,119.8,0.0,2.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
725,1020.8,18.2,17.6,16.1,13.7,96.0,95.0,0.0,34.3,-6.880243e-02,...,-63.5,2319.4,3.5,0.3,25.9,-25.8,25.8,1.6,0.0,39.2
726,1011.7,23.2,18.1,16.0,16.0,78.0,80.0,1.6,25.2,-5.161967e-02,...,-63.5,2747.5,3.2,0.3,29.5,-29.5,29.5,0.0,0.0,0.0
727,1022.7,21.0,18.5,17.0,15.5,92.0,96.0,0.0,21.9,-3.442161e-02,...,-60.8,2651.2,3.1,0.3,29.8,-28.7,28.7,13.4,0.0,5.2
728,1014.4,21.0,20.0,19.7,19.8,94.0,93.0,0.0,39.5,-1.721336e-02,...,-63.2,2459.0,3.4,0.3,27.7,-26.6,26.6,13.4,0.0,5.2


In [61]:
test.to_csv('data/test_prev.csv', index=False)

In [37]:
train = pd.read_csv('data/train_prev.csv')
train.drop(['rainfall', 'prev1_rainfall'], axis=1, inplace=True)

In [38]:
pd.Series(train.columns)[pd.Series(train.columns).apply(lambda x: x not in test.columns)]

Series([], dtype: object)